<a href="https://colab.research.google.com/github/Amatalrahman/EduMind-AI/blob/main/nti_project_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Amatalrahman/EduMind-AI.git
%cd EduMind

Cloning into 'EduMind-AI'...
[Errno 2] No such file or directory: 'EduMind'
/content


In [1]:
!pip install -qU langchain-community langchain-text-splitters pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
from pathlib import Path
import re
import json

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_566/3745420158.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
DATA_DIR = Path("/content/data")

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

SUPPORTED_EXTENSIONS = [".pdf"]

In [5]:
DATA_DIR.mkdir(parents=True, exist_ok=True) # Create data dir if it not exist

In [6]:
def load_pdf(pdf_path):
    """
    Load one PDF and return its pages as LangChain Documents.
    """

    loader = PyPDFLoader(str(pdf_path))
    documents = loader.load()

    return documents

In [7]:
def load_all_pdfs(data_dir):
    """
    Load all PDF files from the given directory.
    """

    pdf_files = sorted(data_dir.glob("*.pdf"))

    if not pdf_files:
        raise FileNotFoundError(
            f"No PDF files found in {data_dir}"
        )

    all_documents = []

    for pdf_file in pdf_files:

        documents = load_pdf(pdf_file)

        all_documents.extend(documents)

        print(
            f"{pdf_file.name}: "
            f"{len(documents)} pages loaded"
        )

    print(f"\nTotal pages loaded: {len(all_documents)}")

    return all_documents

In [8]:
documents = load_all_pdfs(DATA_DIR)

pdf1.pdf: 3 pages loaded

Total pages loaded: 3


In [9]:
def clean_text(text):
    """
    Basic language-independent cleaning.
    Works with Arabic and English.
    """

    text = text.replace("\x00", " ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s+([.,!?;:؟،])", r"\1", text)
    return text.strip()

In [10]:
def preprocess_documents(documents):
    """
    Clean document text while preserving metadata.
    """

    cleaned_documents = []

    for doc in documents:

        cleaned = clean_text(doc.page_content)

        if not cleaned:
            continue

        doc.page_content = cleaned

        cleaned_documents.append(doc)

    return cleaned_documents

In [11]:
documents = preprocess_documents(documents)

print(f"Documents after preprocessing: {len(documents)}")

Documents after preprocessing: 3


In [12]:
def add_document_metadata(documents):
    """
    Add useful metadata without removing
    the metadata provided by the PDF loader.
    """

    for doc in documents:

        source = doc.metadata.get("source", "unknown")

        doc.metadata["file_name"] = Path(source).name
        doc.metadata["file_type"] = "pdf"

        # Convert zero-based page index to 1-based page number
        if "page" in doc.metadata:
            doc.metadata["page_number"] = (
                doc.metadata["page"] + 1
            )

    return documents

In [13]:
documents = add_document_metadata(documents)

In [14]:
documents[0].metadata

{'producer': 'LibreOffice 26.2.4.2 (X86_64)',
 'creator': 'Writer',
 'creationdate': '2026-08-10T23:47:24+03:00',
 'source': '/content/data/pdf1.pdf',
 'total_pages': 3,
 'page': 0,
 'page_label': '1',
 'file_name': 'pdf1.pdf',
 'file_type': 'pdf',
 'page_number': 1}

In [15]:
def create_chunks(
    documents,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
):
    """
    Split documents into overlapping chunks.
    """

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators = [
            "\n\n",
            "\n",
            "؟ ", "؟",
            "? ", "?",
            ". ", ".",
            "!", "؛", "،",
            " ", ""
        ]
    )

    chunks = splitter.split_documents(documents)

    return chunks

In [16]:
chunks = create_chunks(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 4


In [17]:
def add_chunk_metadata(chunks):
    """
    Add a unique ID to every chunk.
    """
    for idx, chunk in enumerate(chunks):
        if "page_number" not in chunk.metadata: # Ensure correct page_number exists
            if "page" in chunk.metadata:
                chunk.metadata["page_number"] = chunk.metadata["page"] + 1
            else:
                chunk.metadata["page_number"] = 1

        file_name = chunk.metadata.get("file_name", "doc")
        page = chunk.metadata.get("page_number")

        chunk.metadata["chunk_id"] = f"{file_name}_p{page}_c{idx}" # (ex: lecture.pdf_p1_c0)

    return chunks

In [18]:
chunks = add_chunk_metadata(chunks)

In [19]:
# Func to print a sample of chunks
def inspect_chunks(chunks, n=5):

    print(f"Total chunks: {len(chunks)}")

    for i, chunk in enumerate(chunks[:n]):

        print("\n" + "=" * 70)

        print(f"Chunk ID: {chunk.metadata.get('chunk_id')}")

        print(
            f"File: "
            f"{chunk.metadata.get('file_name')}"
        )

        print(
            f"Page: "
            f"{chunk.metadata.get('page_number')}"
        )

        print(
            f"Characters: "
            f"{len(chunk.page_content)}"
        )

        print("\nText:")
        print(chunk.page_content[:500])

In [20]:
inspect_chunks(chunks, n=5)

Total chunks: 4

Chunk ID: pdf1.pdf_p1_c0
File: pdf1.pdf
Page: 1
Characters: 993

Text:
Comprehensive Guide to Artificial Intelligence Concepts
1. Introduction to Artificial Intelligence Artificial 
Intelligence (AI) refers to the simulation of human 
intelligence in machines that are programmed to think, learn, 
and solve problems like humans. AI systems leverage 
computer science, data analytics, and domain-specific 
knowledge to create smart solutions capable of processing 
vast amounts of information in real time.
2. Machine Learning Foundations Machine Learning (ML) 
is a core

Chunk ID: pdf1.pdf_p1_c1
File: pdf1.pdf
Page: 1
Characters: 358

Text:
Machines).
• Unsupervised Learning: Algorithms discover hidden 
patterns, groupings, or clusters in unlabeled data (e.g., K-
Means Clustering, Principal Component Analysis).
• Reinforcement Learning: Agents learn optimal actions 
through trial and error by interacting with an 
environment to maximize cumulative rewards (e.g., Q-
Learning

In [21]:
def chunk_statistics(chunks):
"""Display minimum, maximum, and average chunk lengths."""
    lengths = [
        len(chunk.page_content)
        for chunk in chunks
    ]

    print("Number of chunks:", len(lengths))
    print("Minimum length:", min(lengths))
    print("Maximum length:", max(lengths))
    print(
        "Average length:",
        round(sum(lengths) / len(lengths), 2)
    )

In [22]:
chunk_statistics(chunks)

Number of chunks: 4
Minimum length: 358
Maximum length: 993
Average length: 706.25


In [23]:
# Save chunks and metadata into a JSON file
def save_chunks_to_json(chunks, output_path):

    data = []

    for chunk in chunks:

        data.append({
            "text": chunk.page_content,
            "metadata": chunk.metadata
        })

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"Saved to: {output_path}")

In [24]:
save_chunks_to_json(
    chunks,
    "/content/chunks.json"
)

Saved to: /content/chunks.json


In [25]:
ensure_ascii=False

In [26]:
def load_chunks_from_json(file_path):

    from langchain_core.documents import Document

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    chunks = [
        Document(
            page_content=item["text"],
            metadata=item["metadata"]
        )
        for item in data
    ]

    return chunks

In [27]:
chunks = load_chunks_from_json(
    "/content/chunks.json"
)

print(len(chunks))

4


In [28]:
def build_document_chunks(data_dir, chunk_size=1000, chunk_overlap=200):
    """Run the complete document loading, cleaning, and chunking pipeline."""

    documents = load_all_pdfs(data_dir) # load pdf
    documents = preprocess_documents(documents) # clean text
    documents = add_document_metadata(documents) # add metadata
    # split doc into chunks
    chunks = create_chunks(
        documents,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = add_chunk_metadata(chunks) # add unique chunk IDs and metadata

    return chunks

In [29]:
chunks = build_document_chunks(
    DATA_DIR,
    chunk_size=1000,
    chunk_overlap=200
)

pdf1.pdf: 3 pages loaded

Total pages loaded: 3
